# 847. Shortest Path Visiting All Nodes

## Topic Alignment
- This problem represents the Traveling Salesman Problem (TSP) variant on graphs, critical for route optimization in logistics, circuit design, and distributed system coordination.
- Bitmask DP on graphs appears in ML pipeline dependency resolution, model serving optimization across data centers, and distributed training job scheduling.
- The technique is foundational for understanding computational complexity (NP-hard problems) and approximation algorithms in production systems.

## Metadata Summary
- Source: https://leetcode.com/problems/shortest-path-visiting-all-nodes/
- Tags: Dynamic Programming, BFS, Bit Manipulation, Graph, State Compression
- Difficulty: Hard
- Priority: High

## Problem Statement
You have an undirected, connected graph of `n` nodes labeled from `0` to `n - 1`. You are given an array `graph` where `graph[i]` is a list of all the nodes connected with node `i` by an edge.

Return the length of the shortest path that visits every node. You may start and stop at any node, you may revisit nodes multiple times, and you may reuse edges.

**Constraints**:
- `n == graph.length`
- `1 <= n <= 12`
- `0 <= graph[i].length < n`
- `graph[i]` does not contain `i`.
- If `graph[a]` contains `b`, then `graph[b]` contains `a`.
- The input graph is always connected.

## Progressive Hints
- Hint 1: This is a variant of TSP (Traveling Salesman Problem) where we can revisit nodes.
- Hint 2: Use bitmask to represent which nodes have been visited (n <= 12 → 2^12 = 4096 states).
- Hint 3: State: (current_node, visited_mask) represents "at node X, having visited nodes in mask".
- Hint 4: Use BFS to find shortest path (BFS guarantees shortest in unweighted graph).
- Hint 5: Start BFS from all nodes simultaneously (any starting point is valid).
- Hint 6: Goal state: visited_mask == (1 << n) - 1 (all nodes visited).
- Hint 7: Each BFS step: try moving to all neighbors, update visited mask.
- Hint 8: Use (node, mask) as state in visited set to avoid reprocessing same state.

## Solution Overview
Use **BFS with state compression (bitmask DP)**.

**Key Insight**: 
- State = (current_node, visited_nodes_mask)
- BFS explores states in order of path length
- First time we reach all-visited state is the shortest path

**State Definition**:
- `(node, mask)`: currently at `node`, have visited nodes represented by `mask`
- `mask`: bit i is 1 if node i has been visited
- Goal: reach any state with `mask == (1 << n) - 1`

**Algorithm**:
```python
1. Initialize BFS queue with all starting states:
   For each node i: add (i, 1 << i) with distance 0
   
2. BFS:
   - Pop (node, mask, dist) from queue
   - If mask == all_visited: return dist
   - For each neighbor:
     - new_mask = mask | (1 << neighbor)
     - If (neighbor, new_mask) not seen:
       - Mark as seen
       - Add (neighbor, new_mask, dist+1) to queue
```

**Why BFS not DFS?**
- BFS finds shortest path in unweighted graph
- DFS would need to try all paths (exponential)
- BFS with memoization avoids redundant exploration

## Detailed Explanation

### Problem Analysis

**What makes this hard?**
- Must visit ALL nodes (not just find path between two nodes)
- Can start anywhere
- Can revisit nodes and reuse edges
- Need SHORTEST such path

**Difference from TSP**:
- TSP: visit all nodes exactly once, return to start
- This problem: visit all nodes (can revisit), any start/end
- Both are NP-hard in general, but n <= 12 makes bitmask DP feasible

---

### State Space Design

**State components**:
1. **current_node**: which node we're at (0 to n-1)
2. **visited_mask**: which nodes we've visited (bitmask)

**Why this state is sufficient?**
- Knowing current position + visited set determines possible next moves
- Order of visiting doesn't matter for future (only set matters)
- BFS ensures when we reach a state, it's via shortest path

**State space size**: n × 2^n
- n positions × 2^n masks
- For n=12: 12 × 4096 = 49,152 states (very manageable)

---

### Bit Manipulation Details

**Initialize mask for node i**:
```python
mask = 1 << i  # Only bit i is set
```
Example: i=3 → mask = 0b1000 (only node 3 visited)

**Add node j to visited mask**:
```python
new_mask = mask | (1 << j)
```
Example: mask = 0b1010, j=2 → new_mask = 0b1110

**Check if all nodes visited**:
```python
if mask == (1 << n) - 1:
    # All n bits are set
```
Example: n=4 → (1<<4)-1 = 0b1111 (all 4 nodes visited)

**Check if node i is in mask**:
```python
if mask & (1 << i):
    # Node i is visited
```

---

### BFS Strategy

**Why start from all nodes?**
- Problem allows any starting node
- We don't know which start gives shortest path
- Solution: try all, BFS will naturally find the best

**Multi-source BFS**:
```python
queue = deque()
for i in range(n):
    initial_mask = 1 << i
    queue.append((i, initial_mask, 0))  # (node, mask, distance)
```

**State deduplication**:
- Use set to track seen (node, mask) pairs
- If we've seen (node, mask) before, skip it
- BFS guarantees first time seeing a state = shortest path to that state

---

### Why Revisiting Is Allowed

**Example where revisiting is necessary**:
```
Graph: 0 -- 1 -- 2
           |
           3
```

Optimal path from node 0:
- 0 → 1 → 2 (visited: 0,1,2)
- 2 → 1 (revisit!) → 3 (visited: 0,1,2,3)
- Total: 4 edges

Without revisiting:
- Would need to plan perfect route in advance (NP-hard)

With revisiting:
- Can greedily explore, backtrack as needed
- BFS finds optimal with revisiting

---

### Example Walkthrough

**Graph**: `[[1,2,3],[0],[0],[0]]`
```
    1
    |
0 - 2
    |
    3
```
(Actually 0 connects to 1,2,3; 1,2,3 only connect to 0)

**BFS execution**:

**Step 0** (initialization):
- Add (0, 0b0001, 0), (1, 0b0010, 0), (2, 0b0100, 0), (3, 0b1000, 0)

**Step 1** (distance = 0):
- Process (0, 0b0001, 0):
  - Neighbors: 1,2,3
  - Add (1, 0b0011, 1), (2, 0b0101, 1), (3, 0b1001, 1)
- Process (1, 0b0010, 0):
  - Neighbor: 0
  - Add (0, 0b0011, 1)
- Similarly for 2, 3...

**Step 2** (distance = 1):
- Process (1, 0b0011, 1):  # At node 1, visited {0,1}
  - Neighbor: 0
  - Add (0, 0b0011, 2) - already seen, skip
- Process (2, 0b0101, 1):  # At node 2, visited {0,2}
  - Neighbor: 0
  - Add (0, 0b0101, 2) - already seen, skip
- Process (3, 0b1001, 1):  # At node 3, visited {0,3}
  - Neighbor: 0
  - Add (0, 0b1001, 2) - already seen, skip

Actually, from (0, 0b0001), we can reach:
- (1, 0b0011), (2, 0b0101), (3, 0b1001) at distance 1

From any of these, go back to 0:
- From (1, 0b0011) → (0, 0b0011) at distance 2
- From (0, 0b0011), visit 2: (2, 0b0111) at distance 3
- From (2, 0b0111), visit 0: (0, 0b0111) at distance 4
- From (0, 0b0111), visit 3: (3, 0b1111) at distance 5 ✓ (but not optimal)

Better path:
- Start at 0: mask = 0b0001
- Visit 1: mask = 0b0011, dist = 1
- Back to 0: mask = 0b0011, dist = 2
- Visit 2: mask = 0b0111, dist = 3
- Back to 0: mask = 0b0111, dist = 4
- Visit 3: mask = 0b1111, dist = 5

Wait, let's reconsider. If we start from different nodes:

Starting from 0:
- 0 → 1 → 0 → 2 → 0 → 3: total 5 edges

Starting from 1:
- 1 → 0 → 2 → 0 → 3: total 4 edges (better!)

**Answer**: 4

---

### Time Complexity Deep Dive

**State space**: O(n × 2^n)
- n nodes × 2^n possible visited sets

**Transitions per state**: O(degree)
- Each state tries all neighbors
- Average degree ≈ E/n

**Total**: O(n × 2^n × E/n) = O(E × 2^n)
- For dense graph: E = O(n^2) → O(n^2 × 2^n)
- For sparse graph: E = O(n) → O(n × 2^n)

**In practice**: Very efficient for n <= 12
- 12 × 4096 × 12 ≈ 590K operations (milliseconds)

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| BFS + bitmask | O(n × 2^n × E/n) | O(n × 2^n) | Optimal for small n |
| DFS + memo | O(n × 2^n × E/n) | O(n × 2^n) | Same complexity, harder to implement |
| Brute force | O(n!) | O(n) | Try all permutations, impractical |
| MST-based | O(E log V) | O(V) | Incorrect, doesn't visit all nodes |

In [ ]:
from typing import List
from collections import deque

class Solution:
    def shortestPathLength(self, graph: List[List[int]]) -> int:
        """
        BFS with state compression to find shortest path visiting all nodes.
        
        Time: O(n × 2^n × avg_degree)
        Space: O(n × 2^n)
        """
        n = len(graph)
        
        # Edge case: single node
        if n == 1:
            return 0
        
        # Goal: visit all nodes (all bits set)
        all_visited = (1 << n) - 1
        
        # BFS queue: (current_node, visited_mask, distance)
        queue = deque()
        
        # Visited set: (node, mask) pairs we've seen
        seen = set()
        
        # Initialize: start from every node
        for i in range(n):
            mask = 1 << i  # Only node i visited
            queue.append((i, mask, 0))
            seen.add((i, mask))
        
        # BFS
        while queue:
            node, mask, dist = queue.popleft()
            
            # Try all neighbors
            for neighbor in graph[node]:
                # Update visited mask
                new_mask = mask | (1 << neighbor)
                
                # Check if we've visited all nodes
                if new_mask == all_visited:
                    return dist + 1
                
                # If this state not seen before, add to queue
                state = (neighbor, new_mask)
                if state not in seen:
                    seen.add(state)
                    queue.append((neighbor, new_mask, dist + 1))
        
        # Should never reach here if graph is connected
        return -1

In [ ]:
# Test cases
tests = [
    ([[1,2,3],[0],[0],[0]], 4),           # Star graph
    ([[1],[0,2,4],[1,3,4],[2],[1,2]], 4), # More connected
    ([[1],[0]], 1),                       # Two nodes
    ([[]], 0),                            # Single node
    ([[1,2],[0,2],[0,1]], 2),             # Triangle
]

solver = Solution()
for graph, expected in tests:
    result = solver.shortestPathLength(graph)
    assert result == expected, f"Failed for graph={graph}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(n × 2^n × E/n) = O(E × 2^n) where E = number of edges
  - State space: n nodes × 2^n masks
  - Each state explores all neighbors
  - Each state processed at most once (due to seen set)
  - For n=12: ~590K operations
- **Space**: O(n × 2^n) for seen set and queue
  - Seen set stores at most n × 2^n states
  - Queue size at most O(n × 2^n) in worst case
  - For n=12: ~49K states in memory

## Edge Cases & Pitfalls
- **Single node**: Distance is 0 (already visited all)
- **Two nodes connected**: Distance is 1
- **Disconnected graph**: Problem states graph is always connected
- **Complete graph**: Shortest path is n-1 (visit each once)
- **Star graph**: Must revisit center, distance ≈ 2(n-1) - 1
- **Linear graph**: Distance is n-1 (traverse once)
- **Common mistake**: Forgetting to start from all nodes (only starting from 0)
- **Common mistake**: Not checking goal when updating mask (checking only after pop)
- **Common mistake**: Using only mask as state key (need (node, mask) pair)
- **Optimization**: Can check goal immediately after updating mask, return early

## Follow-up Variants
- **Weighted graph**: Edges have weights, find minimum weight path (use Dijkstra instead of BFS)
- **TSP variant**: Must return to starting node (classic TSP)
- **Visit k nodes**: Only need to visit k out of n nodes
- **Directed graph**: Edges have direction, may not be Hamiltonian
- **Time windows**: Each node must be visited within specific time range
- **Multiple agents**: k agents start from different nodes, minimize max time
- **Online version**: Nodes revealed dynamically
- **Approximation**: For larger n, use heuristics (genetic algorithms, simulated annealing)

## Takeaways
- **BFS with bitmask** is the standard approach for small TSP variants.
- **Multi-source BFS** efficiently handles "any starting point" constraint.
- **State = (node, visited_set)** captures all information needed for optimal substructure.
- Allowing revisits makes problem easier than classic TSP (still NP-hard, but more flexible).
- **Bit manipulation** provides O(1) state updates and compact representation.
- Understanding when to use BFS vs Dijkstra (weighted vs unweighted) is crucial.
- This problem demonstrates that some NP-hard problems are tractable for small inputs.
- **State space size** determines feasibility: n <= 12 → 4K states, n <= 20 → 1M states.
- The technique extends to many "visit all" problems: DNA sequencing, robot path planning, etc.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 943 | Find the Shortest Superstring | Bitmask DP, TSP variant |
| LC 996 | Number of Squareful Arrays | Bitmask DP, permutations |
| LC 1125 | Smallest Sufficient Team | Bitmask DP, set cover |
| LC 698 | Partition to K Equal Sum Subsets | Bitmask DP, partitioning |
| LC 1239 | Maximum Length of a Concatenated String | Bitmask for valid states |
| LC 1434 | Number of Ways to Wear Different Hats | Bitmask DP on assignment |